In [0]:
display(dbutils.fs.ls("/Volumes/vb_rag_demo/rag_demo/docs"))

In [0]:
%pip install pypdf python-docx

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# Document ingestion from Volume to Bronze Table

import os
import uuid
from datetime import datetime
import pypdf
from docx import Document

#  Config
CATALOG = "vb_rag_demo"
SCHEMA = "rag_demo"
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/docs"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_documents"

# ------------------------------- Text Extraction Functions -------------------------------

def extract_pdf_text(path: str) -> str:
    try:
        reader = pypdf.PdfReader(path)
        pages = [page.extract_text() or "" for page in reader.pages]
        return "\n".join(pages).strip()
    except Exception as e:
        print(f"Error extracting PDF: {e}")
        return ""

def extract_docx_text(path: str) -> str:
    try:
        doc = Document(path)
        return "\n".join([p.text for p in doc.paragraphs]).strip()
    except Exception as e:
        print(f"Error extracting DOCX: {e}")
        return ""

def extract_txt_text(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()
    except Exception as e:
        print(f"Error extracting TXT: {e}")
        return ""

def extract_text(path: str) -> str:
    lower = path.lower()
    if lower.endswith(".pdf"):
        return extract_pdf_text(path)
    elif lower.endswith(".docx"):
        return extract_docx_text(path)
    elif lower.endswith(".txt"):
        return extract_txt_text(path)
    return ""

# ----------------Ingestion Logic -------------------------------

rows = []

try:
    files = dbutils.fs.ls(VOLUME)

    if not files:
        print(" No files found in volume.")
    else:
        print(f" Found {len(files)} file(s)")

    for item in files:
        try:
            print(f"\n Processing: {item.name}")

            # Copy file from Volume to local /tmp
            local_path = f"/tmp/{item.name}"
            dbutils.fs.cp(item.path, f"file:{local_path}", recurse=False)

            #  Extract text
            text = extract_text(local_path)

            if text:
                rows.append((
                    str(uuid.uuid4()),
                    item.name,
                    item.path,
                    os.path.splitext(item.name)[1].replace(".", "").lower(),
                    text,
                    datetime.utcnow()
                ))
                print(f" Successfully processed: {item.name}")
            else:
                print(f" No text extracted: {item.name}")

        except Exception as e:
            print(f" Error processing {item.name}: {e}")

    # -------------------------------
    # Write to Bronze Table
    # -------------------------------
    if rows:
        df = spark.createDataFrame(rows, [
            "document_id",
            "file_name",
            "file_path",
            "file_type",
            "raw_text",
            "ingested_at"
        ])

        df.write.mode("append").saveAsTable(BRONZE_TABLE)

        print(f"\n SUCCESS: Ingested {len(rows)} document(s) into {BRONZE_TABLE}")
    else:
        print("\n No valid documents to ingest.")

except Exception as e:
    print(f" Fatal error-Processing Files!!!: {e}")